In [ ]:
!pip install -q transformers accelerate tqdm

In [ ]:

import re, string, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU strongly recommended for perplexity.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)


Device: cpu
GPU strongly recommended for perplexity.
Mounted at /content/drive


In [ ]:

BASE_DIR = Path("/content/drive/MyDrive/Linguistics_feature")

AI_DATA_PATH = BASE_DIR / "raid_chatgpt_only.csv"
HUMAN_DATA_PATH = BASE_DIR / "raid_human_matched_to_chatgpt.csv"

AI_OUTPUT_PATH = BASE_DIR / "raid_chatgpt_linguistic_features_v2.csv"
HUMAN_OUTPUT_PATH = BASE_DIR / "raid_human_linguistic_features.csv"

AI_STRUCTURAL_PATH = BASE_DIR / "raid_chatgpt_structural_checkpoint.csv"
HUMAN_STRUCTURAL_PATH = BASE_DIR / "raid_human_structural_checkpoint.csv"

AI_PPL_CHECKPOINT = BASE_DIR / "raid_chatgpt_perplexity_checkpoint.csv"
HUMAN_PPL_CHECKPOINT = BASE_DIR / "raid_human_perplexity_checkpoint.csv"

print(AI_DATA_PATH)
print(HUMAN_DATA_PATH)


/content/drive/MyDrive/Linguistics_feature/raid_chatgpt_only.csv
/content/drive/MyDrive/Linguistics_feature/raid_human_matched_to_chatgpt.csv


In [ ]:

chatgpt_df = pd.read_csv(AI_DATA_PATH)
human_df = pd.read_csv(HUMAN_DATA_PATH)

print("ChatGPT shape:", chatgpt_df.shape)
print("Human shape:", human_df.shape)

print("\nChatGPT columns:")
print(chatgpt_df.columns.tolist())

print("\nHuman columns:")
print(human_df.columns.tolist())

def detect_text_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"No text column found. Available: {df.columns.tolist()}")

AI_TEXT_COL = detect_text_column(chatgpt_df, ["generation", "text", "content"])
HUMAN_TEXT_COL = detect_text_column(human_df, ["text", "generation", "content"])

print("\nAI text column:", AI_TEXT_COL)
print("Human text column:", HUMAN_TEXT_COL)

required_ai = ["id", "source_id", "attack", "domain"]
required_human = ["source_id", "domain"]

missing_ai = [x for x in required_ai if x not in chatgpt_df.columns]
missing_human = [x for x in required_human if x not in human_df.columns]

if missing_ai:
    raise KeyError(f"Missing AI columns: {missing_ai}")
if missing_human:
    raise KeyError(f"Missing human columns: {missing_human}")

chatgpt_df[AI_TEXT_COL] = chatgpt_df[AI_TEXT_COL].astype("string").str.strip()
human_df[HUMAN_TEXT_COL] = human_df[HUMAN_TEXT_COL].astype("string").str.strip()

display(chatgpt_df.head(3))
display(human_df.head(3))


ChatGPT shape: (320904, 11)
Human shape: (13371, 7)

ChatGPT columns:
['id', 'adv_source_id', 'source_id', 'model', 'decoding', 'repetition_penalty', 'attack', 'domain', 'title', 'prompt', 'generation']

Human columns:
['id', 'source_id', 'domain', 'title', 'text', 'model', 'attack']

AI text column: generation
Human text column: text


,id,adv_source_id,source_id,model,decoding,repetition_penalty,attack,domain,title,prompt,generation
0,966c7ba3-c272-431e-a058-73d9d8480e17,966c7ba3-c272-431e-a058-73d9d8480e17,e5e058ce-be2b-459d-af36-32532aaba5ff,chatgpt,greedy,no,none,abstracts,FUTURE-AI: Guiding Principles and Consensus Re...,Write the abstract for the academic paper titl...,Artificial intelligence (AI) has the potential...
1,bd8ef628-ada7-4f49-84b2-456f65a22839,bd8ef628-ada7-4f49-84b2-456f65a22839,e5e058ce-be2b-459d-af36-32532aaba5ff,chatgpt,sampling,no,none,abstracts,FUTURE-AI: Guiding Principles and Consensus Re...,Write the abstract for the academic paper titl...,Artificial intelligence (AI) has the potential...
2,b8ba7698-aca6-4333-b10d-6cd48d5f575b,b8ba7698-aca6-4333-b10d-6cd48d5f575b,f95b107b-d176-4af5-90f7-4d0bb20caf93,chatgpt,greedy,no,none,abstracts,EdgeFlow: Achieving Practical Interactive Segm...,Write the abstract for the academic paper titl...,EdgeFlow is a novel approach to interactive im...


,id,source_id,domain,title,text,model,attack
0,e5e058ce-be2b-459d-af36-32532aaba5ff,e5e058ce-be2b-459d-af36-32532aaba5ff,abstracts,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,human,none
1,f95b107b-d176-4af5-90f7-4d0bb20caf93,f95b107b-d176-4af5-90f7-4d0bb20caf93,abstracts,EdgeFlow: Achieving Practical Interactive Segm...,High-quality training data play a key role in ...,human,none
2,856d8972-9e3d-4544-babc-0fe16f21e04d,856d8972-9e3d-4544-babc-0fe16f21e04d,abstracts,Semi-supervised Contrastive Learning for Label...,The success of deep learning methods in medica...,human,none


In [ ]:

ai_source_ids = set(chatgpt_df["source_id"].dropna().astype(str))
human_source_ids = set(human_df["source_id"].dropna().astype(str))

matched = ai_source_ids & human_source_ids
coverage = 100 * len(matched) / len(ai_source_ids) if ai_source_ids else np.nan

print("Unique AI source_id:", len(ai_source_ids))
print("Unique human source_id:", len(human_source_ids))
print("Matched source_id:", len(matched))
print(f"Human matching coverage: {coverage:.2f}%")

attack_norm = (
    chatgpt_df["attack"].astype("string").fillna("")
    .str.strip().str.lower()
)

original_mask = attack_norm.eq("none")
attacked_mask = ~original_mask

print("\nOriginal AI rows:", int(original_mask.sum()))
print("Attacked AI rows:", int(attacked_mask.sum()))

display(
    chatgpt_df.loc[attacked_mask, "attack"]
    .value_counts(dropna=False)
    .rename_axis("attack")
    .reset_index(name="rows")
)

print("\nHuman domains:")
display(
    human_df["domain"].value_counts()
    .rename_axis("domain").reset_index(name="rows")
)

if "adv_source_id" in chatgpt_df.columns:
    known_ids = set(chatgpt_df["id"].dropna().astype(str))
    adv = chatgpt_df.loc[attacked_mask, "adv_source_id"].dropna().astype(str)
    print("\nAttacked rows with adv_source_id:", len(adv))
    if len(adv):
        print(
            "adv_source_id present among current id values:",
            f"{adv.isin(known_ids).mean()*100:.2f}%"
        )


Unique AI source_id: 13371
Unique human source_id: 13371
Matched source_id: 13371
Human matching coverage: 100.00%

Original AI rows: 26742
Attacked AI rows: 294162


,attack,rows
0,whitespace,26742
1,upper_lower,26742
2,synonym,26742
3,paraphrase,26742
4,number,26742
5,insert_paragraphs,26742
6,perplexity_misspelling,26742
7,homoglyph,26742
8,article_deletion,26742
9,alternative_spelling,26742



Human domains:


,domain,rows
0,books,1781
1,news,1780
2,wiki,1779
3,reddit,1779
4,recipes,1772
5,poetry,1771
6,abstracts,1766
7,reviews,943



Attacked rows with adv_source_id: 294162
adv_source_id present among current id values: 100.00%


In [ ]:

WORD_RE = re.compile(r"\b\w+\b", re.UNICODE)
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")

def tokenize_words(text):
    return WORD_RE.findall(str(text).lower())

def split_sentences(text):
    text = str(text).strip()
    if not text:
        return []
    sents = [s.strip() for s in SENTENCE_RE.split(text) if s.strip()]
    return sents if sents else [text]

def count_syllables(word):
    word = re.sub(r"[^a-z]", "", str(word).lower())
    if not word:
        return 0

    vowels = "aeiouy"
    count = 0
    prev_vowel = False

    for ch in word:
        is_vowel = ch in vowels
        if is_vowel and not prev_vowel:
            count += 1
        prev_vowel = is_vowel

    if word.endswith("e") and count > 1:
        count -= 1

    return max(count, 1)

def extract_structural_features(text):
    if pd.isna(text):
        text = ""
    text = str(text)

    words = tokenize_words(text)
    n_words = len(words)

    sentences = split_sentences(text)
    sent_lengths = [
        len(tokenize_words(s))
        for s in sentences
        if len(tokenize_words(s)) > 0
    ]

    n_sentences = len(sent_lengths)

    lexical_diversity = (
        len(set(words)) / n_words
        if n_words else np.nan
    )

    if sent_lengths:
        avg_sentence_length = float(np.mean(sent_lengths))
        sentence_length_std = float(np.std(sent_lengths, ddof=0))
    else:
        avg_sentence_length = np.nan
        sentence_length_std = np.nan

    burstiness = (
        sentence_length_std / avg_sentence_length
        if np.isfinite(avg_sentence_length)
        and avg_sentence_length > 0
        else np.nan
    )

    punctuation_count = sum(ch in string.punctuation for ch in text)
    punctuation_density = (
        100 * punctuation_count / len(text)
        if len(text) else np.nan
    )

    if n_words and n_sentences:
        syllables = sum(count_syllables(w) for w in words)
        readability_flesch = (
            206.835
            - 1.015 * (n_words / n_sentences)
            - 84.6 * (syllables / n_words)
        )
    else:
        readability_flesch = np.nan

    return {
        "word_count": n_words,
        "lexical_diversity": lexical_diversity,
        "avg_sentence_length": avg_sentence_length,
        "sentence_length_std": sentence_length_std,
        "burstiness": burstiness,
        "punctuation_density": punctuation_density,
        "readability_flesch": readability_flesch,
    }

print(
    extract_structural_features(
        "This is a test sentence. This second sentence is somewhat longer!"
    )
)


{'word_count': 11, 'lexical_diversity': 0.7272727272727273, 'avg_sentence_length': 5.5, 'sentence_length_std': 0.5, 'burstiness': 0.09090909090909091, 'punctuation_density': 3.076923076923077, 'readability_flesch': 70.50704545454548}


In [ ]:

FEATURES_7 = [
    "word_count",
    "lexical_diversity",
    "avg_sentence_length",
    "sentence_length_std",
    "burstiness",
    "punctuation_density",
    "readability_flesch",
]

def add_structural_features(df, text_col, desc):
    rows = [
        extract_structural_features(text)
        for text in tqdm(df[text_col].tolist(), desc=desc)
    ]
    return pd.concat(
        [df.reset_index(drop=True), pd.DataFrame(rows)],
        axis=1
    )

if AI_STRUCTURAL_PATH.exists():
    print("Loading AI structural checkpoint")
    chatgpt_features = pd.read_csv(AI_STRUCTURAL_PATH)
else:
    chatgpt_features = add_structural_features(
        chatgpt_df, AI_TEXT_COL, "AI structural features"
    )
    chatgpt_features.to_csv(AI_STRUCTURAL_PATH, index=False)

if HUMAN_STRUCTURAL_PATH.exists():
    print("Loading human structural checkpoint")
    human_features = pd.read_csv(HUMAN_STRUCTURAL_PATH)
else:
    human_features = add_structural_features(
        human_df, HUMAN_TEXT_COL, "Human structural features"
    )
    human_features.to_csv(HUMAN_STRUCTURAL_PATH, index=False)

print("AI structural shape:", chatgpt_features.shape)
print("Human structural shape:", human_features.shape)

display(chatgpt_features[FEATURES_7].head())
display(human_features[FEATURES_7].head())


Loading AI structural checkpoint
Loading human structural checkpoint
AI structural shape: (320904, 18)
Human structural shape: (13371, 14)


,word_count,lexical_diversity,avg_sentence_length,sentence_length_std,burstiness,punctuation_density,readability_flesch
0,125,0.688000,25.000000,5.329165,0.213167,2.550478,-8.044000
1,110,0.636364,22.000000,3.286335,0.149379,2.078240,1.461364
2,94,0.702128,23.500000,4.716991,0.200723,1.152738,-3.317500
3,130,0.661538,21.666667,4.346135,0.200591,1.498929,3.929487
4,146,0.664384,20.857143,6.311635,0.302613,1.442308,8.352671


,word_count,lexical_diversity,avg_sentence_length,sentence_length_std,burstiness,punctuation_density,readability_flesch
0,211,0.620853,35.166667,8.193832,0.233000,3.266667,-2.068645
1,149,0.711409,16.555556,3.803832,0.229762,3.134182,27.644534
2,224,0.549107,24.888889,9.643011,0.387442,1.990391,24.458492
3,281,0.576512,28.100000,7.803204,0.277694,2.257709,22.059407
4,238,0.512605,23.800000,9.410632,0.395405,2.050581,38.004891


In [ ]:

GPT2_MODEL_NAME = "gpt2"
PPL_MAX_LENGTH = 512
PPL_BATCH_SIZE = 8 if DEVICE == "cuda" else 2

gpt2_tokenizer = GPT2TokenizerFast.from_pretrained(GPT2_MODEL_NAME)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

gpt2_model = GPT2LMHeadModel.from_pretrained(
    GPT2_MODEL_NAME
).to(DEVICE)

gpt2_model.eval()

print("Model:", GPT2_MODEL_NAME)
print("Max tokens:", PPL_MAX_LENGTH)
print("Batch size:", PPL_BATCH_SIZE)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: gpt2
Max tokens: 512
Batch size: 8


In [ ]:

@torch.no_grad()
def calculate_perplexity_batch(texts):
    texts = ["" if pd.isna(x) else str(x) for x in texts]

    enc = gpt2_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=PPL_MAX_LENGTH,
    )

    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)

    logits = gpt2_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    ).logits

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    shift_mask = attention_mask[:, 1:].contiguous()

    token_loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        reduction="none",
    ).view(shift_labels.shape)

    token_loss = token_loss * shift_mask.float()
    valid_tokens = shift_mask.sum(dim=1)

    mean_loss = token_loss.sum(dim=1) / valid_tokens.clamp(min=1)
    ppl = torch.exp(mean_loss)

    ppl = torch.where(
        valid_tokens > 0,
        ppl,
        torch.full_like(ppl, float("nan"))
    )

    return ppl.detach().cpu().numpy().astype(float)

sample_ppl = calculate_perplexity_batch([
    "The cat sat on the mat.",
    "This is a somewhat longer example sentence used to test the language model."
])
print(sample_ppl)


[ 90.2437973  149.85044861]


In [ ]:
def compute_perplexity_optimized(
    df,
    text_col,
    checkpoint_path,
    dataset_name,
    batch_size=32,
    save_every=500
):
    checkpoint_path = Path(checkpoint_path)

    work = df.reset_index(drop=True).copy()
    work["_feature_row_id"] = np.arange(len(work), dtype=int)

    # -----------------------------------------
    # 1. Load existing row-level checkpoint
    # -----------------------------------------
    if checkpoint_path.exists():

        ckpt = pd.read_csv(checkpoint_path)

        ckpt = (
            ckpt[
                ["_feature_row_id", "perplexity"]
            ]
            .drop_duplicates("_feature_row_id")
        )

        work = work.merge(
            ckpt,
            on="_feature_row_id",
            how="left"
        )

        print(
            f"{dataset_name} checkpoint loaded:",
            work["perplexity"].notna().sum(),
            "/",
            len(work)
        )

    else:

        work["perplexity"] = np.nan

    # -----------------------------------------
    # 2. Reuse PPL for duplicate texts already
    #    calculated in the checkpoint
    # -----------------------------------------

    completed_texts = (
        work[
            work["perplexity"].notna()
        ][
            [text_col, "perplexity"]
        ]
        .drop_duplicates(
            subset=[text_col]
        )
    )

    if len(completed_texts):

        ppl_lookup = dict(
            zip(
                completed_texts[text_col],
                completed_texts["perplexity"]
            )
        )

        missing_mask = work["perplexity"].isna()

        work.loc[
            missing_mask,
            "perplexity"
        ] = (
            work.loc[
                missing_mask,
                text_col
            ]
            .map(ppl_lookup)
        )

    print(
        "After duplicate reuse:",
        work["perplexity"].notna().sum(),
        "/",
        len(work)
    )

    # -----------------------------------------
    # 3. Only calculate UNIQUE remaining texts
    # -----------------------------------------

    remaining = work[
        work["perplexity"].isna()
    ][
        ["_feature_row_id", text_col]
    ].copy()

    unique_remaining = (
        remaining
        .drop_duplicates(
            subset=[text_col]
        )
        .copy()
    )

    print(
        "Remaining rows:",
        len(remaining)
    )

    print(
        "Unique remaining texts:",
        len(unique_remaining)
    )

    print(
        "Duplicate calculations avoided:",
        len(remaining) - len(unique_remaining)
    )

    # -----------------------------------------
    # 4. Sort by approximate text length
    #
    # Similar-length documents in each batch
    # reduce wasted padding.
    # -----------------------------------------

    unique_remaining["_text_length"] = (
        unique_remaining[text_col]
        .fillna("")
        .astype(str)
        .str.len()
    )

    unique_remaining = (
        unique_remaining
        .sort_values("_text_length")
        .reset_index(drop=True)
    )

    text_to_ppl = {}

    processed = 0

    # -----------------------------------------
    # 5. Calculate perplexity
    # -----------------------------------------

    for start in tqdm(
        range(
            0,
            len(unique_remaining),
            batch_size
        ),
        desc=f"{dataset_name} GPT-2 perplexity"
    ):

        batch = unique_remaining.iloc[
            start:start + batch_size
        ]

        texts = batch[text_col].tolist()

        try:

            ppl_values = (
                calculate_perplexity_batch(
                    texts
                )
            )

        except RuntimeError as e:

            # GPU-memory fallback
            if (
                DEVICE == "cuda"
                and
                "out of memory" in str(e).lower()
            ):

                print(
                    "\nGPU memory limit hit.",
                    "Retrying this batch individually."
                )

                torch.cuda.empty_cache()
                gc.collect()

                ppl_values = np.array([
                    calculate_perplexity_batch(
                        [text]
                    )[0]
                    for text in texts
                ])

            else:
                raise

        # Save text -> perplexity
        for text, ppl in zip(
            texts,
            ppl_values
        ):

            text_to_ppl[text] = float(ppl)

        processed += len(batch)

        # -------------------------------------
        # Regular checkpoint
        # -------------------------------------

        if (
            processed >= save_every
            or
            start + batch_size >= len(unique_remaining)
        ):

            missing_mask = (
                work["perplexity"].isna()
            )

            work.loc[
                missing_mask,
                "perplexity"
            ] = (
                work.loc[
                    missing_mask,
                    text_col
                ]
                .map(text_to_ppl)
            )

            work[
                [
                    "_feature_row_id",
                    "perplexity"
                ]
            ].to_csv(
                checkpoint_path,
                index=False
            )

            processed = 0

    # -----------------------------------------
    # 6. Final duplicate mapping
    # -----------------------------------------

    missing_mask = work[
        "perplexity"
    ].isna()

    work.loc[
        missing_mask,
        "perplexity"
    ] = (
        work.loc[
            missing_mask,
            text_col
        ]
        .map(text_to_ppl)
    )

    # -----------------------------------------
    # 7. Final checkpoint
    # -----------------------------------------

    work[
        [
            "_feature_row_id",
            "perplexity"
        ]
    ].to_csv(
        checkpoint_path,
        index=False
    )

    print(
        "\nCompleted:",
        work["perplexity"].notna().sum(),
        "/",
        len(work)
    )

    return (
        work
        .sort_values("_feature_row_id")
        .drop(
            columns="_feature_row_id"
        )
        .reset_index(drop=True)
    )

In [ ]:
print("Total rows:", len(chatgpt_features))

print(
    "Unique generations:",
    chatgpt_features["generation"].nunique()
)

print(
    "Duplicate generations:",
    chatgpt_features["generation"].duplicated().sum()
)

Total rows: 320904
Unique generations: 290618
Duplicate generations: 30286


In [ ]:
PPL_BATCH_SIZE = 32

chatgpt_features = compute_perplexity_optimized(
    df=chatgpt_features,
    text_col=AI_TEXT_COL,
    checkpoint_path=AI_PPL_CHECKPOINT,
    dataset_name="ChatGPT",
    batch_size=PPL_BATCH_SIZE,
    save_every=1000
)

ChatGPT checkpoint loaded: 84168 / 320904
After duplicate reuse: 84168 / 320904
Remaining rows: 236736
Unique remaining texts: 217872
Duplicate calculations avoided: 18864


ChatGPT GPT-2 perplexity:   0%|          | 0/6809 [00:00<?, ?it/s]


Completed: 320904 / 320904


In [ ]:

human_features = compute_perplexity_optimized(
    df=human_features,
    text_col=HUMAN_TEXT_COL,
    checkpoint_path=HUMAN_PPL_CHECKPOINT,
    dataset_name="Human",
    batch_size=PPL_BATCH_SIZE,
    save_every=1000
)

print(
    "Human perplexity complete:",
    human_features["perplexity"].notna().sum(),
    "/",
    len(human_features)
)


Human remaining: 13371


Human GPT-2 perplexity:   0%|          | 0/1672 [00:00<?, ?it/s]

Human perplexity complete: 13371 / 13371


In [ ]:
FEATURE_COLS = FEATURES_7 + ["perplexity"]

def quality_table(df, dataset):
    rows = []
    for feature in FEATURE_COLS:
        x = pd.to_numeric(df[feature], errors="coerce")
        rows.append({
            "dataset": dataset,
            "feature": feature,
            "rows": len(df),
            "missing": int(x.isna().sum()),
            "mean": x.mean(),
            "std": x.std(),
            "min": x.min(),
            "max": x.max(),
        })
    return pd.DataFrame(rows)

display(
    pd.concat([
        quality_table(chatgpt_features, "ChatGPT"),
        quality_table(human_features, "Human"),
    ], ignore_index=True).round(4)
)

print("Eight features:", FEATURE_COLS)


,dataset,feature,rows,missing,mean,std,min,max
0,ChatGPT,word_count,320904,0,351.4647,354.4136,0.0000,2894.0000
1,ChatGPT,lexical_diversity,320904,1,0.5232,0.1693,0.0042,1.0000
2,ChatGPT,avg_sentence_length,320904,1,128.5588,398.2094,3.0000,2894.0000
3,ChatGPT,sentence_length_std,320904,1,5.6147,3.9829,0.0000,139.0000
4,ChatGPT,burstiness,320904,1,0.3390,0.3193,0.0000,2.9410
5,ChatGPT,punctuation_density,320904,0,2.5470,1.0591,0.0000,50.0234
6,ChatGPT,readability_flesch,320904,1,-52.4152,392.8352,-2815.0581,198.7150
7,ChatGPT,perplexity,320904,0,18.2978,14.1279,1.0238,447.0394
8,Human,word_count,13371,0,302.4249,322.1930,0.0000,12762.0000
9,Human,lexical_diversity,13371,1,0.5687,0.0843,0.0020,1.0000


Eight features: ['word_count', 'lexical_diversity', 'avg_sentence_length', 'sentence_length_std', 'burstiness', 'punctuation_density', 'readability_flesch', 'perplexity']


In [ ]:
final_ai_sources = set(
    chatgpt_features["source_id"].dropna().astype(str)
)
final_human_sources = set(
    human_features["source_id"].dropna().astype(str)
)

print(
    "Matched source IDs in final feature files:",
    len(final_ai_sources & final_human_sources)
)
print(
    "AI source IDs missing from final human file:",
    len(final_ai_sources - final_human_sources)
)

attack_final = (
    chatgpt_features["attack"].astype("string")
    .fillna("").str.strip().str.lower()
)

print(
    "\nOriginal AI feature rows:",
    int(attack_final.eq("none").sum())
)
print(
    "Attacked AI feature rows:",
    int((~attack_final.eq("none")).sum())
)

ai_preview = [
    x for x in
    ["id", "adv_source_id", "source_id", "attack", "domain"] + FEATURE_COLS
    if x in chatgpt_features.columns
]

human_preview = [
    x for x in
    ["id", "source_id", "domain"] + FEATURE_COLS
    if x in human_features.columns
]

display(chatgpt_features[ai_preview].head(10))
display(human_features[human_preview].head(10))


Matched source IDs in final feature files: 13371
AI source IDs missing from final human file: 0

Original AI feature rows: 26742
Attacked AI feature rows: 294162


,id,adv_source_id,source_id,attack,domain,word_count,lexical_diversity,avg_sentence_length,sentence_length_std,burstiness,punctuation_density,readability_flesch,perplexity
0,966c7ba3-c272-431e-a058-73d9d8480e17,966c7ba3-c272-431e-a058-73d9d8480e17,e5e058ce-be2b-459d-af36-32532aaba5ff,none,abstracts,125,0.688000,25.000000,5.329165,0.213167,2.550478,-8.044000,15.127154
1,bd8ef628-ada7-4f49-84b2-456f65a22839,bd8ef628-ada7-4f49-84b2-456f65a22839,e5e058ce-be2b-459d-af36-32532aaba5ff,none,abstracts,110,0.636364,22.000000,3.286335,0.149379,2.078240,1.461364,16.072073
2,b8ba7698-aca6-4333-b10d-6cd48d5f575b,b8ba7698-aca6-4333-b10d-6cd48d5f575b,f95b107b-d176-4af5-90f7-4d0bb20caf93,none,abstracts,94,0.702128,23.500000,4.716991,0.200723,1.152738,-3.317500,27.342155
3,4936ee69-adab-4225-a39f-170bde100cab,4936ee69-adab-4225-a39f-170bde100cab,f95b107b-d176-4af5-90f7-4d0bb20caf93,none,abstracts,130,0.661538,21.666667,4.346135,0.200591,1.498929,3.929487,19.430410
4,9b233912-6936-47ed-b3c4-52f858fbb7f6,9b233912-6936-47ed-b3c4-52f858fbb7f6,856d8972-9e3d-4544-babc-0fe16f21e04d,none,abstracts,146,0.664384,20.857143,6.311635,0.302613,1.442308,8.352671,21.126175
5,b1a656fd-8e5e-46d0-a983-16239b3989b5,b1a656fd-8e5e-46d0-a983-16239b3989b5,856d8972-9e3d-4544-babc-0fe16f21e04d,none,abstracts,142,0.598592,23.666667,5.763872,0.243544,1.422764,5.868263,21.298332
6,c99c448f-8e64-46dc-97b2-7327e178198a,c99c448f-8e64-46dc-97b2-7327e178198a,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,none,abstracts,153,0.568627,21.857143,4.015277,0.183705,1.812689,23.744118,19.352945
7,b6d4d98d-7a1d-4293-a726-6db2ce4ae8d8,b6d4d98d-7a1d-4293-a726-6db2ce4ae8d8,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,none,abstracts,147,0.598639,21.000000,4.985694,0.237414,2.426694,22.650612,20.548683
8,b5ad65b6-011b-4eef-96cb-89421c37c4f0,b5ad65b6-011b-4eef-96cb-89421c37c4f0,72c41b8d-0069-4886-b734-a4000ffca286,none,abstracts,113,0.690265,22.600000,3.382307,0.149660,1.293661,10.203965,39.436928
9,8ec46839-8d25-4b28-b0a6-6a31406d27e7,8ec46839-8d25-4b28-b0a6-6a31406d27e7,72c41b8d-0069-4886-b734-a4000ffca286,none,abstracts,146,0.643836,24.333333,3.986087,0.163812,1.552795,24.525708,17.153721


,id,source_id,domain,word_count,lexical_diversity,avg_sentence_length,sentence_length_std,burstiness,punctuation_density,readability_flesch,perplexity
0,e5e058ce-be2b-459d-af36-32532aaba5ff,e5e058ce-be2b-459d-af36-32532aaba5ff,abstracts,211,0.620853,35.166667,8.193832,0.233000,3.266667,-2.068645,29.249554
1,f95b107b-d176-4af5-90f7-4d0bb20caf93,f95b107b-d176-4af5-90f7-4d0bb20caf93,abstracts,149,0.711409,16.555556,3.803832,0.229762,3.134182,27.644534,33.153347
2,856d8972-9e3d-4544-babc-0fe16f21e04d,856d8972-9e3d-4544-babc-0fe16f21e04d,abstracts,224,0.549107,24.888889,9.643011,0.387442,1.990391,24.458492,38.394405
3,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,fbc8a5ea-90fa-47b8-8fa7-73dd954f1524,abstracts,281,0.576512,28.100000,7.803204,0.277694,2.257709,22.059407,39.000351
4,72c41b8d-0069-4886-b734-a4000ffca286,72c41b8d-0069-4886-b734-a4000ffca286,abstracts,238,0.512605,23.800000,9.410632,0.395405,2.050581,38.004891,48.838028
5,72fe360b-cce6-4daf-b66a-1d778f5964f8,72fe360b-cce6-4daf-b66a-1d778f5964f8,abstracts,199,0.587940,19.900000,6.847627,0.344102,1.627907,31.465646,52.768314
6,df594cf4-9a0c-4488-bcb3-68f41e2d5a16,df594cf4-9a0c-4488-bcb3-68f41e2d5a16,abstracts,208,0.586538,20.800000,4.686150,0.225296,2.496715,8.388385,45.945053
7,853c0e51-7dd5-4bb5-8286-e4aa8820173b,853c0e51-7dd5-4bb5-8286-e4aa8820173b,abstracts,161,0.683230,26.833333,6.094168,0.227112,2.717901,27.214073,48.846588
8,1649f195-8f98-4c79-92b6-54a5ca9261fa,1649f195-8f98-4c79-92b6-54a5ca9261fa,abstracts,194,0.530928,21.555556,6.308920,0.292682,2.800000,37.124152,50.327049
9,5e23ab14-b85f-48e8-9aa3-15452e73524e,5e23ab14-b85f-48e8-9aa3-15452e73524e,abstracts,251,0.617530,22.818182,7.553665,0.331037,3.001876,30.990083,37.673622
